# Lab 11 - Apple Brand Monitor Semantic Search

This notebook adapts the Lab 11 embeddings and semantic-search workflow to the `social_media_brand_monitor` project. The original lab prompt uses movies as the example domain, but this submission applies the same embedding, vector-database, filtering, and search concepts to Apple brand-monitoring documents.


## Project Scope

The semantic-search system is built on Apple-related articles and structured records stored in `data/processed/cleaned/cleaned_data.csv`.

- semantic documents are built from `title`, `overview`, `description`, `content`, and `primary_keyword`
- ChromaDB metadata uses real project fields such as `source`, `document_type`, `language`, `mention_year`, and `primary_keyword`
- keyword, semantic, and hybrid search are evaluated on Apple-related queries


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings import (
    BrandEmbedder,
    ChromaBrandStore,
    build_filter_examples,
    compare_search,
    compare_search_side_by_side,
    compare_synonym_query_pairs,
    cosine_similarity_matrix,
    cosine_similarity_scores,
    dot_product_scores,
    euclidean_distances,
    hybrid_search,
    keyword_search,
    prepare_embedding_dataframe,
    reciprocal_rank_fusion,
    run_embeddings_pipeline,
    semantic_search,
)

CLEANED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned" / "cleaned_data.csv"
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "processed" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEANED_CSV_PATH)
prepared_df = prepare_embedding_dataframe(df)
embedder = BrandEmbedder()

prepared_df[["_id", "title", "source", "document_type", "language", "mention_year", "primary_keyword"]].head()

,_id,title,source,document_type,language,mention_year,primary_keyword
0,69ce9d979d3427d3654d3276,News: Nach Rücktritt - »Mein erster großer Feh...,newsapi_Apple_page_2.json,json,unknown,2026,apple
1,69ce9d979d3427d3654d3278,Apple doesnt lawsuit,sample.csv,csv,unknown,2026,apple
2,69ce9d979d3427d3654d3279,Apple faces lawsuit,sample.xml,xml,unknown,2026,apple
3,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,json,unknown,2026,apple
4,69ed2b5ef527cb7a1c1159eb,Apple ville sätta stopp för föreningslogga i Ö...,newsapi_Apple_page_1.json,json,unknown,2026,apple


## 1. Embedding Generation

This section demonstrates how text is converted into dense numeric vectors. Instead of using movie descriptions, I use Apple-related text samples and then apply the same process to the real dataset.


In [2]:
sample_texts = [
    "Apple launches a new iPhone with improved camera features and AI tools.",
    "Tim Cook discusses Apple's long-term product strategy and research spending.",
    "Apple TV adds new sports and streaming content for subscribers.",
    "A smartphone review compares camera quality, battery life, and performance.",
    "Executives explain the company's innovation roadmap and future plans.",
]

sample_embeddings = embedder.encode(sample_texts)
sample_embeddings.shape

INFO: Loading embedding model | model_name=sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO: Generated embeddings | texts=5 | dimensions=384


(5, 384)

The output shape should be `(5, 384)`. That means five sample texts were embedded, and each one was converted into a 384-dimensional vector. This confirms that the embedding model is working correctly and producing fixed-length semantic representations.


In [3]:
dataset_documents, dataset_embeddings = embedder.encode_dataframe(
    prepared_df.head(10),
    extra_columns=["description", "content", "source", "document_type", "language"],
)

print("documents:", len(dataset_documents))
print("embedding shape:", dataset_embeddings.shape)
print(dataset_documents[0][:400])

INFO: Generated embeddings | texts=10 | dimensions=384


documents: 10
embedding shape: (10, 384)
Title: News: Nach Rücktritt - »Mein erster großer Fehler«: Tim Cook blickt auf seine Zeit als Apple-CEO zurück | Overview: Nach über 15 Jahren hört Tim Cook als Apple-CEO auf und nutzt die Gegenwart auch, um diese Zeit Revue passieren zu lassen. Dabei gesteht er seinen »ersten großen Fehler« seiner Amtszeit ein. | Keywords: apple | Description: Nach über 15 Jahren hört Tim Cook als Apple-CEO auf u


The dataframe example shows that each Apple mention is first converted into one combined semantic-search document before embedding. This is important because the model can use title, summary, keyword, and source context together instead of relying on only one text field.


## 2. Similarity Measures

Now I compare cosine similarity, dot product, and Euclidean distance. Higher cosine similarity and dot product values indicate stronger semantic similarity, while lower Euclidean distance indicates stronger similarity.


In [4]:
similarity_matrix = cosine_similarity_matrix(sample_embeddings)
pd.DataFrame(similarity_matrix, index=range(1, 6), columns=range(1, 6)).round(3)

,1,2,3,4,5
1,1.000,0.317,0.325,0.439,0.238
2,0.317,1.000,0.281,0.219,0.423
3,0.325,0.281,1.000,0.121,0.098
4,0.439,0.219,0.121,1.000,0.067
5,0.238,0.423,0.098,0.067,1.000


In [5]:
query_text = "Apple executive leadership and long-term company strategy"
query_embedding = embedder.encode(query_text)

cosine_scores = cosine_similarity_scores(query_embedding, sample_embeddings)
dot_scores = dot_product_scores(query_embedding, sample_embeddings)
euclidean_scores = euclidean_distances(query_embedding, sample_embeddings)

similarity_df = pd.DataFrame({
    "text": sample_texts,
    "cosine_similarity": cosine_scores,
    "dot_product": dot_scores,
    "euclidean_distance": euclidean_scores,
}).sort_values("cosine_similarity", ascending=False)

similarity_df.round(4)

INFO: Generated embeddings | texts=1 | dimensions=384


,text,cosine_similarity,dot_product,euclidean_distance
1,Tim Cook discusses Apple's long-term product s...,0.6518,0.6518,0.8346
4,Executives explain the company's innovation ro...,0.4726,0.4726,1.0270
0,Apple launches a new iPhone with improved came...,0.3067,0.3067,1.1776
2,Apple TV adds new sports and streaming content...,0.2313,0.2313,1.2399
3,"A smartphone review compares camera quality, b...",0.1558,0.1558,1.2993


The leadership and strategy query should rank the Tim Cook and innovation-roadmap texts near the top. This shows that embeddings capture meaning rather than relying only on exact word overlap.


## 3. ChromaDB Setup and Population

This section creates a persistent ChromaDB collection stored on disk in `data/embeddings/chroma_db`. The collection name is `data`, and cosine similarity is used as the ranking space.


In [6]:
store = ChromaBrandStore()
store.get_or_create_collection(reset=True)
inserted_count = store.add_documents(prepared_df)

print("inserted_count:", inserted_count)
print("collection_count:", store.count())

INFO: Connected to ChromaDB | path=data\embeddings\chroma_db
INFO: Deleted existing ChromaDB collection | name=data
INFO: Ready ChromaDB collection | name=data
INFO: Loading embedding model | model_name=sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO: Generated embeddings | texts=101 | dimensions=384
INFO: Upserted documents into ChromaDB | rows=101 | collection=data


inserted_count: 101
collection_count: 101


The vector database is persistent, so after the initial population step the Apple documents can be searched again without rebuilding everything from scratch. Metadata is stored together with embeddings so semantic retrieval can later be narrowed by structured filters.


## 4. Semantic Search with ChromaDB

The next examples demonstrate single-query and multi-query semantic retrieval using the same ChromaDB collection.


In [7]:
single_query_results = store.query_to_dataframe(
    query_text="Apple leadership and product strategy",
    n_results=5,
)

single_query_results[[
    "query_text", "rank", "_id", "distance", "source", "document_type", "language", "mention_year", "primary_keyword"
]].round(4)

INFO: Generated embeddings | texts=1 | dimensions=384


INFO: Queried ChromaDB | queries=1 | n_results=5


,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,Apple leadership and product strategy,1,69fc8502ef3b485ee4bc2b6d,0.4525,newsapi_Apple_page_3.json,json,unknown,2026,apple
1,Apple leadership and product strategy,2,69ed2e00f527cb7a1c115a90,0.4622,newsapi_Apple_page_1.json,json,unknown,2026,apple
2,Apple leadership and product strategy,3,69fc8502ef3b485ee4bc2b46,0.5114,newsapi_Apple_page_1.json,json,unknown,2026,apple
3,Apple leadership and product strategy,4,69ed2b5ef527cb7a1c115a08,0.5629,newsapi_Apple_page_3.json,json,unknown,2026,vision
4,Apple leadership and product strategy,5,69fc8502ef3b485ee4bc2b41,0.5729,newsapi_Apple_page_1.json,json,unknown,2026,apple


In [8]:
multi_query_results = store.multi_query_to_dataframe(
    query_texts=[
        "Apple product launch and iPhone features",
        "Tim Cook leadership and company strategy",
    ],
    n_results=3,
)

multi_query_results[["query_text", "rank", "_id", "distance", "source", "primary_keyword"]].round(4)

INFO: Generated embeddings | texts=2 | dimensions=384


INFO: Queried ChromaDB | queries=2 | n_results=3


,query_text,rank,_id,distance,source,primary_keyword
0,Apple product launch and iPhone features,1,69f79537c7d32a5f5b914592,0.5114,newsapi_Apple_page_3.json,apple
1,Apple product launch and iPhone features,2,69ed2b5ff527cb7a1c115a33,0.5348,sample.csv,apple
2,Apple product launch and iPhone features,3,69ed2b5ef527cb7a1c115a08,0.5837,newsapi_Apple_page_3.json,vision
3,Tim Cook leadership and company strategy,1,69ed2b5ef527cb7a1c1159f0,0.5656,newsapi_Apple_page_1.json,apple
4,Tim Cook leadership and company strategy,2,69ce9d979d3427d3654d3276,0.6145,newsapi_Apple_page_2.json,apple
5,Tim Cook leadership and company strategy,3,69ed2e00f527cb7a1c115a90,0.6629,newsapi_Apple_page_1.json,apple


The multi-query example shows that ChromaDB can process several natural-language queries in a single API call. This is useful when testing search behavior across multiple themes without repeating setup work.


## 5. Metadata Filtering

Here I combine semantic search with metadata filters. Because this project is not a movie dataset, I use real Apple-monitor fields such as `language`, `mention_year`, `document_type`, `source`, and `primary_keyword`.


In [9]:
filters = build_filter_examples()
filters

{'eq_language_unknown': {'language': {'$eq': 'unknown'}},
 'gte_2025': {'mention_year': {'$gte': 2025}},
 'in_document_types': {'document_type': {'$in': ['json',
    'csv',
    'web_scrape']}},
 'and_keyword_year': {'$and': [{'primary_keyword': {'$eq': 'apple'}},
   {'mention_year': {'$gte': 2025}}]},
 'or_language_source': {'$or': [{'language': {'$eq': 'unknown'}},
   {'source': {'$eq': 'sample.csv'}}]}}

In [10]:
filter_demo_results = {
    "eq_language_unknown": store.query_to_dataframe("Apple news coverage", n_results=5, where=filters["eq_language_unknown"]),
    "gte_2025": store.query_to_dataframe("recent Apple reports", n_results=5, where=filters["gte_2025"]),
    "in_document_types": store.query_to_dataframe("Apple mentions from structured files", n_results=5, where=filters["in_document_types"]),
    "and_keyword_year": store.query_to_dataframe("recent Apple coverage", n_results=5, where=filters["and_keyword_year"]),
    "or_language_source": store.query_to_dataframe("Apple sample or unknown-language content", n_results=5, where=filters["or_language_source"]),
}

for name, result in filter_demo_results.items():
    print(f"\n=== {name} ===")
    display(result[[
        "query_text", "rank", "_id", "distance", "source", "document_type", "language", "mention_year", "primary_keyword"
    ]].head())

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5



=== eq_language_unknown ===


,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,Apple news coverage,1,69f79537c7d32a5f5b914579,0.512865,newsapi_Apple_page_1.json,json,unknown,2026,apple
1,Apple news coverage,2,69ed2b5ef527cb7a1c115a0c,0.541812,newsapi_Apple_page_3.json,json,unknown,2026,apple
2,Apple news coverage,3,69fc8502ef3b485ee4bc2b60,0.555362,newsapi_Apple_page_3.json,json,unknown,2026,apple
3,Apple news coverage,4,69ed2b5ef527cb7a1c1159f1,0.561484,newsapi_Apple_page_1.json,json,unknown,2026,apple
4,Apple news coverage,5,69fc86b4ef3b485ee4bc2b74,0.566210,newsapi_Apple_page_1.json,json,unknown,2026,apple



=== gte_2025 ===


,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,recent Apple reports,1,69fc8502ef3b485ee4bc2b6d,0.457069,newsapi_Apple_page_3.json,json,unknown,2026,apple
1,recent Apple reports,2,69fc8502ef3b485ee4bc2b46,0.498404,newsapi_Apple_page_1.json,json,unknown,2026,apple
2,recent Apple reports,3,69ed2b5ef527cb7a1c115a05,0.522560,newsapi_Apple_page_3.json,json,unknown,2026,apple
3,recent Apple reports,4,69ed2b5ff527cb7a1c115a27,0.530784,apple_ratings_large.csv,csv,en,2026,apple
4,recent Apple reports,5,69ed2b5ff527cb7a1c115a1b,0.530785,apple_ratings_large.csv,csv,en,2026,apple


...

,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,Apple mentions from structured files,1,69ed2b5ff527cb7a1c115a33,0.521003,sample.csv,csv,unknown,2026,apple
1,Apple mentions from structured files,2,69ed2b5ff527cb7a1c115a27,0.562514,apple_ratings_large.csv,csv,en,2026,apple
2,Apple mentions from structured files,3,69ed2b5ff527cb7a1c115a1f,0.562514,apple_ratings_large.csv,csv,en,2026,apple
3,Apple mentions from structured files,4,69ed2b5ff527cb7a1c115a2b,0.562514,apple_ratings_large.csv,csv,en,2026,apple
4,Apple mentions from structured files,5,69ed2b5ff527cb7a1c115a2f,0.562514,apple_ratings_large.csv,csv,en,2026,apple



=== and_keyword_year ===


,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,recent Apple coverage,1,69fc8502ef3b485ee4bc2b6d,0.525907,newsapi_Apple_page_3.json,json,unknown,2026,apple
1,recent Apple coverage,2,69f79537c7d32a5f5b914594,0.543786,newsapi_Apple_page_3.json,json,unknown,2026,apple
2,recent Apple coverage,3,69f79537c7d32a5f5b914579,0.544569,newsapi_Apple_page_1.json,json,unknown,2026,apple
3,recent Apple coverage,4,69fc8502ef3b485ee4bc2b46,0.549219,newsapi_Apple_page_1.json,json,unknown,2026,apple
4,recent Apple coverage,5,69fc8502ef3b485ee4bc2b47,0.550802,newsapi_Apple_page_1.json,json,unknown,2026,apple



=== or_language_source ===


,query_text,rank,_id,distance,source,document_type,language,mention_year,primary_keyword
0,Apple sample or unknown-language content,1,69ed2b5ff527cb7a1c115a33,0.426562,sample.csv,csv,unknown,2026,apple
1,Apple sample or unknown-language content,2,69ce9d979d3427d3654d3279,0.528276,sample.xml,xml,unknown,2026,apple
2,Apple sample or unknown-language content,3,69ce9d979d3427d3654d3278,0.545564,sample.csv,csv,unknown,2026,apple
3,Apple sample or unknown-language content,4,69ed2b5ef527cb7a1c115a03,0.585500,newsapi_Apple_page_2.json,json,unknown,2026,apple
4,Apple sample or unknown-language content,5,69ed2b5ff527cb7a1c115a34,0.586684,sample.csv,csv,unknown,2026,apple


These examples demonstrate at least four Chroma filter operators: `$eq`, `$gte`, `$in`, `$and`, and `$or`. This confirms that the semantic-search system supports both meaning-based retrieval and structured narrowing at the same time.


## 6. Keyword, Semantic, and Hybrid Search

This section compares traditional keyword search, semantic search, and hybrid search built with Reciprocal Rank Fusion.


In [11]:
comparison_results = compare_search(
    query="Apple leadership and product strategy",
    df=prepared_df,
    store=store,
    n_results=5,
)

for name, result in comparison_results.items():
    print(f"\n=== {name.upper()} ===")
    columns = [c for c in [
        "_id", "title", "source", "document_type", "language", "mention_year",
        "keyword_score", "semantic_score", "distance", "rrf_score"
    ] if c in result.columns]
    display(result[columns].head())

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=10



=== KEYWORD ===


,_id,title,source,document_type,language,mention_year,keyword_score
0,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,json,unknown,2026,8
1,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,json,unknown,2026,8
2,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,json,unknown,2026,7
3,69ed2b5ef527cb7a1c1159f9,How to watch Major League Baseball games Frida...,newsapi_Apple_page_2.json,json,unknown,2026,7
4,69ed2b5ef527cb7a1c115a0c,What To Watch This Weekend: New Shows And Movi...,newsapi_Apple_page_3.json,json,unknown,2026,7



=== SEMANTIC ===


,_id,title,source,document_type,language,mention_year,semantic_score,distance
0,69fc8502ef3b485ee4bc2b6d,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,json,unknown,2026,0.547471,0.452529
1,69ed2e00f527cb7a1c115a90,Will incoming CEO John Ternus help realize App...,newsapi_Apple_page_1.json,json,unknown,2026,0.537785,0.462215
2,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,json,unknown,2026,0.488565,0.511435
3,69ed2b5ef527cb7a1c115a08,How John Ternus could finally fulfill Steve Jo...,newsapi_Apple_page_3.json,json,unknown,2026,0.437148,0.562852
4,69fc8502ef3b485ee4bc2b41,Tech Brand Outlet Deals at eBay: Up to 60% off...,newsapi_Apple_page_1.json,json,unknown,2026,0.427059,0.572941



=== HYBRID ===


,_id,title,source,document_type,language,mention_year,keyword_score,semantic_score,distance,rrf_score
0,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,json,unknown,2026,NaN,0.488565,0.511435,0.032002
1,69f79537c7d32a5f5b914592,Two New Ultra Products to be Released by Apple,newsapi_Apple_page_3.json,json,unknown,2026,NaN,0.425585,0.574415,0.029644
2,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,json,unknown,2026,8.0,NaN,NaN,0.016393
3,69fc8502ef3b485ee4bc2b6d,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,json,unknown,2026,NaN,0.547471,0.452529,0.016393
4,69ed2e00f527cb7a1c115a90,Will incoming CEO John Ternus help realize App...,newsapi_Apple_page_1.json,json,unknown,2026,NaN,0.537785,0.462215,0.016129


In [12]:
side_by_side_df = compare_search_side_by_side(
    query="Apple leadership and product strategy",
    df=prepared_df,
    store=store,
    n_results=3,
)

side_by_side_df[[
    "search_method", "_id", "title", "source", "keyword_score", "semantic_score", "distance", "rrf_score"
]].head(9)

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=3
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=6


,search_method,_id,title,source,keyword_score,semantic_score,distance,rrf_score
0,keyword,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,8.0,NaN,NaN,NaN
1,keyword,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,8.0,NaN,NaN,NaN
2,keyword,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,7.0,NaN,NaN,NaN
3,semantic,69fc8502ef3b485ee4bc2b6d,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,NaN,0.547471,0.452529,NaN
4,semantic,69ed2e00f527cb7a1c115a90,Will incoming CEO John Ternus help realize App...,newsapi_Apple_page_1.json,NaN,0.537785,0.462215,NaN
5,semantic,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,NaN,0.488565,0.511435,NaN
6,hybrid,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,NaN,0.488565,0.511435,0.032002
7,hybrid,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,8.0,NaN,NaN,0.016393
8,hybrid,69fc8502ef3b485ee4bc2b6d,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,NaN,0.547471,0.452529,0.016393


The comparison shows a useful trade-off. Keyword search is good when the same words appear directly in the text, but semantic search retrieves conceptually related Apple leadership and strategy articles even when the wording changes. Hybrid search combines both strengths using Reciprocal Rank Fusion.


## 7. Synonym Query Pair Evaluation

To test consistency, I compare synonym-style queries and measure result overlap for keyword search versus semantic search.


In [13]:
query_pairs = [
    ("Apple product launch", "Apple product release"),
    ("Tim Cook leadership", "Apple executive strategy"),
    ("iPhone features", "smartphone capabilities"),
]

overlap_df = compare_synonym_query_pairs(
    query_pairs=query_pairs,
    df=prepared_df,
    store=store,
    n_results=5,
)

overlap_df

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5


,query_a,query_b,keyword_overlap_count,keyword_overlap_ratio,semantic_overlap_count,semantic_overlap_ratio
0,Apple product launch,Apple product release,3,0.428571,3,0.428571
1,Tim Cook leadership,Apple executive strategy,1,0.125000,3,0.428571
2,iPhone features,smartphone capabilities,0,0.000000,1,0.111111


The overlap results show that semantic search usually handles synonyms and paraphrases more consistently than keyword search. This is especially visible when the second query uses different wording, such as `executive strategy` instead of `Tim Cook leadership`, or `smartphone capabilities` instead of `iPhone features`.


## 8. Keyword vs Semantic Search Analysis

Semantic search performs better when the user expresses an idea using synonyms, paraphrases, or broader concepts. In this project, that means a query about leadership, innovation, or product strategy can still retrieve relevant Apple documents even if the exact query words do not appear in the text.

Keyword search performs better when the user needs exact matches such as article titles, specific names, IDs, or literal phrases. For example, if I search a precise product or article phrase, keyword matching is often more direct and easier to interpret.

Hybrid search is useful because it combines the precision of keyword search with the recall of semantic search. In practice, that makes hybrid search a strong default choice for the Apple brand-monitor system.


## 9. Analytical Questions

The final section uses the completed semantic-search system to answer real analytical questions about the Apple dataset.


### Question 1

**Question:** Which documents are most strongly related to Apple leadership and company strategy?

**Query used:** `Apple leadership and company strategy`


In [14]:
q1_results = store.query_to_dataframe(
    query_text="Apple leadership and company strategy",
    n_results=5,
)

q1_results[["query_text", "rank", "title", "source", "mention_year", "primary_keyword", "distance"]].round(4)

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5


,query_text,rank,title,source,mention_year,primary_keyword,distance
0,Apple leadership and company strategy,1,Will incoming CEO John Ternus help realize App...,newsapi_Apple_page_1.json,2026,apple,0.4699
1,Apple leadership and company strategy,2,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,2026,apple,0.4759
2,Apple leadership and company strategy,3,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,2026,apple,0.5831
3,Apple leadership and company strategy,4,How John Ternus could finally fulfill Steve Jo...,newsapi_Apple_page_3.json,2026,vision,0.5906
4,Apple leadership and company strategy,5,Tim Cook admits his biggest mistake and proude...,newsapi_Apple_page_1.json,2026,apple,0.5907


This result shows which Apple documents are closest to the theme of leadership and strategic direction. If articles about Tim Cook, executive changes, or long-term investment plans appear near the top, that suggests the dataset contains meaningful coverage of Apple management and corporate strategy.


### Question 2

**Question:** What recent Apple coverage appears when I focus on recent years only?

**Query used:** `recent Apple reports`

**Filter used:** `mention_year >= 2025`


In [15]:
q2_results = store.query_to_dataframe(
    query_text="recent Apple reports",
    n_results=5,
    where={"mention_year": {"$gte": 2025}},
)

q2_results[["query_text", "rank", "title", "source", "document_type", "mention_year", "distance"]].round(4)

INFO: Generated embeddings | texts=1 | dimensions=384


INFO: Queried ChromaDB | queries=1 | n_results=5


,query_text,rank,title,source,document_type,mention_year,distance
0,recent Apple reports,1,Apple's R&D investments top 10% of sales as AI...,newsapi_Apple_page_3.json,json,2026,0.4571
1,recent Apple reports,2,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,json,2026,0.4984
2,recent Apple reports,3,How Apple Filmed its Flashy MacBook Neo Video ...,newsapi_Apple_page_3.json,json,2026,0.5226
3,recent Apple reports,4,Apple iPhone review,apple_ratings_large.csv,csv,2026,0.5308
4,recent Apple reports,5,Apple iPhone review,apple_ratings_large.csv,csv,2026,0.5308


This filtered search shows how semantic retrieval can be narrowed to recent coverage only. That makes the system useful for trend-focused analysis, because I can search by meaning while still restricting the results to a relevant time window.


### Question 3

**Question:** Which Apple-related documents from structured file sources are closest to product and feature discussions?

**Query used:** `Apple product launch and iPhone features`

**Filter used:** `document_type in [json, csv, web_scrape]`


In [16]:
q3_results = store.query_to_dataframe(
    query_text="Apple product launch and iPhone features",
    n_results=5,
    where={"document_type": {"$in": ["json", "csv", "web_scrape"]}},
)

q3_results[["query_text", "rank", "title", "source", "document_type", "primary_keyword", "distance"]].round(4)

INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5


,query_text,rank,title,source,document_type,primary_keyword,distance
0,Apple product launch and iPhone features,1,Two New Ultra Products to be Released by Apple,newsapi_Apple_page_3.json,json,apple,0.5114
1,Apple product launch and iPhone features,2,Apple launches new iPhone,sample.csv,csv,apple,0.5348
2,Apple product launch and iPhone features,3,How John Ternus could finally fulfill Steve Jo...,newsapi_Apple_page_3.json,json,vision,0.5837
3,Apple product launch and iPhone features,4,Apple Mother's Day Deals at Best Buy: Up to 50...,newsapi_Apple_page_1.json,json,apple,0.6328
4,Apple product launch and iPhone features,5,Výjimečný iPhone bude opravdu jiný. Apple má m...,newsapi_Apple_page_1.json,json,apple,0.6349


This question shows that the search system works end to end with metadata filtering. It can retrieve semantically related Apple product discussions while also limiting the search to selected source types. That is useful when comparing how different ingestion channels contribute to the dataset.


## 10. Pipeline Integration Check

The embeddings system was also integrated into the project pipeline. The following cell runs the reusable embeddings pipeline entrypoint and saves a comparison CSV for later inspection.


In [17]:
pipeline_result = run_embeddings_pipeline(CLEANED_CSV_PATH)

print("inserted_count:", pipeline_result["inserted_count"])
print("collection_count:", pipeline_result["collection_count"])
print("comparison_path:", pipeline_result["comparison_path"])

pipeline_result["comparison_df"][[
    "search_method", "_id", "title", "source", "keyword_score", "semantic_score", "distance", "rrf_score"
]].head()

INFO: Connected to ChromaDB | path=data\embeddings\chroma_db
INFO: Ready ChromaDB collection | name=data
INFO: Generated embeddings | texts=101 | dimensions=384
INFO: Upserted documents into ChromaDB | rows=101 | collection=data
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=5
INFO: Generated embeddings | texts=1 | dimensions=384
INFO: Queried ChromaDB | queries=1 | n_results=10
INFO: Embeddings pipeline finished | rows=101 | inserted=101 | collection_count=101 | comparison_output=data\processed\embeddings\search_comparison_demo.csv


inserted_count: 101
collection_count: 101
comparison_path: data\processed\embeddings\search_comparison_demo.csv


,search_method,_id,title,source,keyword_score,semantic_score,distance,rrf_score
0,keyword,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,8.0,NaN,NaN,NaN
1,keyword,69fc8502ef3b485ee4bc2b46,MacBook Neo Won’t Let Chromebooks Or Windows L...,newsapi_Apple_page_1.json,8.0,NaN,NaN,NaN
2,keyword,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,7.0,NaN,NaN,NaN
3,keyword,69ed2b5ef527cb7a1c1159f9,How to watch Major League Baseball games Frida...,newsapi_Apple_page_2.json,7.0,NaN,NaN,NaN
4,keyword,69ed2b5ef527cb7a1c115a0c,What To Watch This Weekend: New Shows And Movi...,newsapi_Apple_page_3.json,7.0,NaN,NaN,NaN


This confirms that the semantic-search workflow is not only demonstrated in the notebook but also integrated into the reusable project pipeline.
